In [0]:
%pip install seaborn

In [0]:
%restart_python

In [0]:

# =======================================================================
# Step 4 - Question 1: Demand Intermittency & Syntetos-Boylan Classification
# =======================================================================
# Purpose:
# 1. Measure the global share of zero-sales days (sparsity of retail demand).
# 2. Categorize each series into one of 4 demand quadrants:
#    - Smooth: Regular demand timing, low variance in non-zero quantity
#    - Intermittent: Infrequent demand timing, low variance in quantity
#    - Erratic: Regular demand timing, high variance in quantity
#    - Lumpy: Infrequent demand timing, high variance in quantity (hardest to forecast)
# 3. Determine why standard metrics like MAPE fail and justify RMSSE / WAPE.
# =======================================================================






from pyspark.sql import functions as F

b = spark.table("workspace.m5.base")

print(
    "share of zero days:",
    b.select(F.avg((F.col("sales") == 0).cast("int"))).first()[0],
)

# Syntetos–Boylan classification, per series
# ADI  = average interval between non-zero sales  (days / non-zero days)
# CV²  = squared coefficient of variation of the NON-ZERO sales
per_series = (
    b.groupBy("id", "dept_id", "store_id")
    .agg(
        F.count("*").alias("days"),
        F.sum((F.col("sales") > 0).cast("int")).alias("nz_days"),
        F.avg(F.when(F.col("sales") > 0, F.col("sales"))).alias("nz_mean"),
        F.stddev(F.when(F.col("sales") > 0, F.col("sales"))).alias("nz_std"),
    )
    .withColumn("adi", F.col("days") / F.col("nz_days"))
    .withColumn("cv2", F.pow(F.col("nz_std") / F.col("nz_mean"), 2))
    .withColumn(
        "demand_class",
        F.when((F.col("adi") < 1.32) & (F.col("cv2") < 0.49), "smooth")
        .when((F.col("adi") < 1.32), "erratic")
        .when((F.col("cv2") < 0.49), "intermittent")
        .otherwise("lumpy"),
    )
)
per_series.write.mode("overwrite").saveAsTable("workspace.m5.series_profile")

spark.table("workspace.m5.series_profile").groupBy("demand_class").count().show()

In [0]:
# Department-level breakdown of demand classes
spark.table("workspace.m5.series_profile").groupBy("dept_id").pivot(
    "demand_class"
).count().fillna(0).show()

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F

# Aggregate daily units per category
# Using date_format / dayofweek from 'date' because 'wday' was not kept in base, and using 'snap'
daily = (
    b.withColumn("wday", F.dayofweek("date"))  # Spark dayofweek: 1=Sunday, 7=Saturday
     .groupBy("date", "cat_id", "wday", "snap")
     .agg(F.sum("sales").alias("units"))
     .orderBy("date")
     .toPandas()
)

daily["date"] = pd.to_datetime(daily["date"])
daily["day_of_month"] = daily["date"].dt.day
daily["month_day"] = daily["date"].dt.strftime("%m-%d")

# Plot 4 seasonal patterns directly in Databricks
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# (a) 5-year daily series trend by category
sns.lineplot(data=daily, x="date", y="units", hue="cat_id", ax=axes[0, 0], alpha=0.8)
axes[0, 0].set_title("(a) 5-Year Daily Sales Trend by Category")
axes[0, 0].set_ylabel("Total Units Sold")

# (b) Average units by day of week
sns.barplot(data=daily, x="wday", y="units", hue="cat_id", ax=axes[0, 1], ci=None)
axes[0, 1].set_title("(b) Average Sales by Day of Week")
axes[0, 1].set_xlabel("Day of Week (1=Sun, 2=Mon, ..., 7=Sat)")

# (c) Day of month pattern with SNAP marked
sns.lineplot(data=daily, x="day_of_month", y="units", hue="cat_id", style="snap", ax=axes[1, 0], ci=None)
axes[1, 0].set_title("(c) Day of Month Pattern & SNAP Effect")
axes[1, 0].set_xlabel("Day of Month")

# (d) Christmas drop (Dec 20 - Dec 30)
dec_window = daily[(daily["date"].dt.month == 12) & (daily["date"].dt.day.between(20, 30))]
sns.lineplot(data=dec_window, x="month_day", y="units", hue="cat_id", marker="o", ax=axes[1, 1], ci=None)
axes[1, 1].set_title("(d) Christmas Drop (Dec 20 - Dec 30)")
axes[1, 1].set_xlabel("Date (MM-DD)")

plt.tight_layout()
plt.show()

In [0]:
# =======================================================================
# Step 4 - Question 3: Who drives the volume and revenue? (Pareto Analysis)
# =======================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------
# 1. Compute 28-day Revenue per Series in Spark
# -----------------------------------------------------------------------
b = spark.table("workspace.m5.base")

# Calculate revenue over the last 28 days of available history (d_num > 1941 - 28)
rev_spark = (
    b.filter("d_num > 1941 - 28")
    .groupBy("id", "item_id", "dept_id", "cat_id", "store_id")
    .agg(
        F.sum(F.col("sales") * F.col("sell_price")).alias("revenue_28d"),
        F.sum("sales").alias("units_28d")
    )
)

# Join with series_profile to analyze demand characteristics of top earners
series_profile = spark.table("workspace.m5.series_profile").select("id", "demand_class", "adi", "cv2")
rev_with_profile = rev_spark.join(series_profile, on="id", how="left")

# Save as table for later use in reporting and forecast slicing
rev_with_profile.write.mode("overwrite").saveAsTable("workspace.m5.series_revenue_28d")

# -----------------------------------------------------------------------
# 2. Collect Summary Data for Pareto Curve (Sorted by Revenue)
# -----------------------------------------------------------------------
# Spark aggregation rule: collect only sorted scalar revenue values (~30k floats)
df_rev = (
    rev_with_profile.select("id", "dept_id", "demand_class", "revenue_28d", "units_28d")
    .orderBy(F.col("revenue_28d").desc())
    .toPandas()
)

# Compute cumulative percentages
df_rev["cum_revenue"] = df_rev["revenue_28d"].cumsum()
total_rev = df_rev["revenue_28d"].sum()
df_rev["cum_rev_share"] = df_rev["cum_revenue"] / total_rev
df_rev["series_rank"] = np.arange(1, len(df_rev) + 1)
df_rev["cum_series_share"] = df_rev["series_rank"] / len(df_rev)

# Key metric: Share of series generating 80% of total revenue
idx_80 = (df_rev["cum_rev_share"] >= 0.80).idxmax()
series_share_80 = df_rev.loc[idx_80, "cum_series_share"]
count_80 = df_rev.loc[idx_80, "series_rank"]

print("=" * 60)
print(f"Total 28-day Revenue: ${total_rev:,.2f}")
print(f"Total Series Count:   {len(df_rev):,}")
print(f"80% of revenue is generated by: {series_share_80 * 100:.2f}% of series ({count_80:,} items)")
print("=" * 60)

# -----------------------------------------------------------------------
# 3. Plot: Pareto Curve (Revenue Concentration)
# -----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df_rev["cum_series_share"] * 100, df_rev["cum_rev_share"] * 100, label="Revenue Concentration", color="#1f77b4", lw=2)
ax.axhline(80, color="gray", linestyle="--", alpha=0.7)
ax.axvline(series_share_80 * 100, color="red", linestyle="--", alpha=0.8, label=f"80% Rev cutoff ({series_share_80 * 100:.1f}% series)")

ax.set_title("Pareto Curve: Cumulative Revenue Distribution (Last 28 Days)")
ax.set_xlabel("% of Total Series (Sorted Descending by Revenue)")
ax.set_ylabel("% of Cumulative Revenue")
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend()
plt.tight_layout()
plt.show()

# -----------------------------------------------------------------------
# 4. Top-300 High Value Items vs. Demand Classification
# -----------------------------------------------------------------------
top_300 = df_rev.head(300)
top_300_rev_share = top_300["revenue_28d"].sum() / total_rev

print(f"\nTop 300 series account for {top_300_rev_share * 100:.2f}% of total revenue.")
print("\nDemand class breakdown among Top 300 items:")
print(top_300["demand_class"].value_counts(normalize=True).round(4) * 100)

print("\nDemand class breakdown across ALL 30,490 series:")
print(df_rev["demand_class"].value_counts(normalize=True).round(4) * 100)